In [1]:
!pip install -q --upgrade torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.4 MB/s eta 0:00:00a 0:00:01


In [2]:
import torch
import transformers
import peft
import torchao

print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT        :", peft.__version__)
print("TorchAO     :", torchao.__version__)
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

PyTorch     : 2.10.0+cu128
Transformers: 5.0.0
PEFT        : 0.19.1
TorchAO     : 0.16.0
GPU         : Tesla T4


In [3]:
from datasets import load_dataset
import json

dataset = load_dataset(
    "Project-AgML/AgroBench",
    split="train"
)

did_dataset = dataset.filter(
    lambda x: json.loads(x["raw_metadata"]).get("source") == "did"
)

did_split = did_dataset.train_test_split(
    test_size=0.20,
    seed=42
)

train_dataset = did_split["train"]
test_dataset = did_split["test"]

print("Train:", len(train_dataset))
print("Test :", len(test_dataset))
print("First train ID:", train_dataset[0]["id"])
print("First test ID :", test_dataset[0]["id"])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/650M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/840M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/629M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4342 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4342 [00:00<?, ? examples/s]

Train: 1201
Test : 301
First train ID: agrobench_png_169
First test ID : agrobench_png_1236


In [4]:
from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor
)

model_id = "Qwen/Qwen3-VL-2B-Instruct"

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [5]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 6,422,528 || all params: 2,133,954,560 || trainable%: 0.3010


In [7]:
import io
from PIL import Image

def collate_fn(examples):

    full_texts = []
    prompt_texts = []
    images = []

    for sample in examples:

        image = Image.open(
            io.BytesIO(sample["images"][0]["bytes"])
        ).convert("RGB")

        question = sample["messages"][0]["content"][1]["text"]
        answer = sample["messages"][1]["content"][0]["text"]

        user_messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": question}
                ]
            }
        ]

        full_messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": question}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": answer}
                ]
            }
        ]

        prompt_text = processor.apply_chat_template(
            user_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        full_text = processor.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False
        )

        prompt_texts.append(prompt_text)
        full_texts.append(full_text)
        images.append(image)

    batch = processor(
        text=full_texts,
        images=images,
        padding=True,
        return_tensors="pt"
    )

    labels = batch["input_ids"].clone()

    labels[
        batch["attention_mask"] == 0
    ] = -100

    for i, prompt_text in enumerate(prompt_texts):

        prompt_inputs = processor(
            text=[prompt_text],
            images=[images[i]],
            padding=False,
            return_tensors="pt"
        )

        prompt_length = prompt_inputs["input_ids"].shape[1]

        labels[i, :prompt_length] = -100

    batch["labels"] = labels

    return batch

In [8]:
test_batch = collate_fn([train_dataset[0]])

print("Input shape:", test_batch["input_ids"].shape)
print(
    "Training tokens:",
    (test_batch["labels"] != -100).sum().item()
)

Input shape: torch.Size([1, 468])
Training tokens: 12


In [9]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qwen3vl_agrobench_did_lora_r16_2epochs",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=1e-4,
    num_train_epochs=2,

    logging_steps=10,
    save_strategy="epoch",

    fp16=True,
    remove_unused_columns=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn
)

train_result = trainer.train()

Step,Training Loss
10,1.602092
20,0.283682
30,0.225598
40,0.316591
50,0.229128
60,0.252076
70,0.291629
80,0.251927
90,0.272126
100,0.190210


In [10]:
print("Final training loss:", train_result.training_loss)

Final training loss: 0.18975469259625655


In [11]:
save_path = "/kaggle/working/qwen3vl_agrobench_did_lora_r16_2epochs"

model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print("Saved:", save_path)

Saved: /kaggle/working/qwen3vl_agrobench_did_lora_r16_2epochs


In [12]:
import shutil

zip_path = shutil.make_archive(
    "/kaggle/working/qwen3vl_agrobench_did_lora_r16_2epochs",
    "zip",
    save_path
)

print("ZIP:", zip_path)

ZIP: /kaggle/working/qwen3vl_agrobench_did_lora_r16_2epochs.zip


In [13]:
import os

zip_path = "/kaggle/working/qwen3vl_agrobench_did_lora_r16_2epochs.zip"

print("Exists:", os.path.exists(zip_path))
print("Size (MB):", os.path.getsize(zip_path) / (1024**2))

Exists: True
Size (MB): 160.34127521514893


In [14]:
import zipfile

with zipfile.ZipFile(zip_path, "r") as z:
    print("ZIP is valid.")
    print(z.namelist())

ZIP is valid.
['checkpoint-301/', 'checkpoint-602/', 'chat_template.jinja', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json', 'checkpoint-602/training_args.bin', 'checkpoint-602/rng_state.pth', 'checkpoint-602/scaler.pt', 'checkpoint-602/README.md', 'checkpoint-602/adapter_model.safetensors', 'checkpoint-602/trainer_state.json', 'checkpoint-602/adapter_config.json', 'checkpoint-602/optimizer.pt', 'checkpoint-602/scheduler.pt', 'checkpoint-301/training_args.bin', 'checkpoint-301/rng_state.pth', 'checkpoint-301/scaler.pt', 'checkpoint-301/README.md', 'checkpoint-301/adapter_model.safetensors', 'checkpoint-301/trainer_state.json', 'checkpoint-301/adapter_config.json', 'checkpoint-301/optimizer.pt', 'checkpoint-301/scheduler.pt']


In [15]:
import os

adapter_path = "/kaggle/input/datasets/iftekharackerman/qwen3vl-agrobench-did-lora-r16-2epochs"

print(os.listdir(adapter_path))

['adapter_model.safetensors', 'adapter_config.json', 'README.md', 'tokenizer.json', 'checkpoint-602', 'tokenizer_config.json', 'checkpoint-301', 'chat_template.jinja', 'processor_config.json']


In [16]:
import torch

from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor
)

from peft import PeftModel

model_id = "Qwen/Qwen3-VL-2B-Instruct"

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(
    adapter_path
)

model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

model.eval()

print("r=16, 2-epoch LoRA model restored successfully!")

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

r=16, 2-epoch LoRA model restored successfully!


In [17]:
import io
import re
import torch
import pandas as pd

from PIL import Image


def parse_options(question):
    matches = re.findall(
        r'([A-E])\.\s*(.+?)(?=\n[A-E]\.|$)',
        question,
        re.DOTALL
    )

    return {
        letter: text.strip()
        for letter, text in matches
    }


def get_correct_option(question, ground_truth):
    options = parse_options(question)

    for letter, option_text in options.items():
        if option_text.casefold() == ground_truth.strip().casefold():
            return letter

    return None


def match_prediction_to_option(question, prediction):
    options = parse_options(question)

    pred = prediction.strip().casefold()

    for letter, option_text in options.items():
        if pred == option_text.casefold():
            return letter

    for letter, option_text in options.items():
        if option_text.casefold() in pred:
            return letter

    return None


controlled_r16_results = []

model.eval()

for i, sample in enumerate(test_dataset):

    question = sample["messages"][0]["content"][1]["text"]
    ground_truth = sample["messages"][1]["content"][0]["text"]

    correct_letter = get_correct_option(
        question,
        ground_truth
    )

    image = Image.open(
        io.BytesIO(sample["images"][0]["bytes"])
    ).convert("RGB")

    controlled_question = (
        question
        + "\n\nSelect the correct disease from the options above. "
          "Respond with only the exact disease name from the selected option. "
          "Do not provide any explanation."
    )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image
                },
                {
                    "type": "text",
                    "text": controlled_question
                }
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        if isinstance(v, torch.Tensor)
        else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False
        )

    generated_ids_trimmed = generated_ids[
        :,
        inputs["input_ids"].shape[1]:
    ]

    raw_prediction = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()

    predicted_letter = match_prediction_to_option(
        question,
        raw_prediction
    )

    correct = predicted_letter == correct_letter

    controlled_r16_results.append(
        {
            "id": sample["id"],
            "ground_truth": ground_truth,
            "correct_letter": correct_letter,
            "prediction": raw_prediction,
            "predicted_letter": predicted_letter,
            "correct": correct
        }
    )

    if (i + 1) % 25 == 0 or i == 0:

        running_accuracy = (
            sum(x["correct"] for x in controlled_r16_results)
            / len(controlled_r16_results)
            * 100
        )

        print(
            f"{i+1:03d}/{len(test_dataset)}"
            f" | Running Accuracy: {running_accuracy:.2f}%"
        )

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


001/301 | Running Accuracy: 100.00%
025/301 | Running Accuracy: 72.00%
050/301 | Running Accuracy: 74.00%
075/301 | Running Accuracy: 74.67%
100/301 | Running Accuracy: 76.00%
125/301 | Running Accuracy: 73.60%
150/301 | Running Accuracy: 74.67%
175/301 | Running Accuracy: 70.86%
200/301 | Running Accuracy: 71.50%
225/301 | Running Accuracy: 69.78%
250/301 | Running Accuracy: 70.00%
275/301 | Running Accuracy: 67.64%
300/301 | Running Accuracy: 67.33%


In [18]:
df_controlled_r16 = pd.DataFrame(
    controlled_r16_results
)

controlled_r16_accuracy = (
    df_controlled_r16["correct"].mean()
    * 100
)

print("\nControlled r=16, 2-Epoch Fine-Tuned Model Results")
print("------------------------------------------------")

print(
    "Correct:",
    df_controlled_r16["correct"].sum()
)

print(
    "Total:",
    len(df_controlled_r16)
)

print(
    f"Accuracy: {controlled_r16_accuracy:.2f}%"
)

print(
    "Unmatched predictions:",
    df_controlled_r16[
        "predicted_letter"
    ].isna().sum()
)

df_controlled_r16.to_csv(
    "/kaggle/working/qwen3vl_did_controlled_r16_2epoch_eval_301.csv",
    index=False
)


Controlled r=16, 2-Epoch Fine-Tuned Model Results
------------------------------------------------
Correct: 202
Total: 301
Accuracy: 67.11%
Unmatched predictions: 0


In [19]:
wrong = df_controlled_r16[
    df_controlled_r16["correct"] == False
]

correct = df_controlled_r16[
    df_controlled_r16["correct"] == True
]

print("Correct predictions:", len(correct))
print("Wrong predictions:", len(wrong))

Correct predictions: 202
Wrong predictions: 99


In [20]:
wrong[
    [
        "id",
        "ground_truth",
        "prediction",
        "correct_letter",
        "predicted_letter"
    ]
].head(20)

,id,ground_truth,prediction,correct_letter,predicted_letter
3,agrobench_png_584,Mosaic,Bacterial leaf blight and rot,E,A
4,agrobench_png_1273,Phytophthora blight,Powdery mildew,C,D
5,agrobench_png_582,Anthracnose and charcoal spot,Black rot,B,D
10,agrobench_png_811,Red stele,Strawberry lethal decline,C,D
20,agrobench_png_1191,Sclerotinia blight,Stunt,A,D
21,agrobench_png_407,Collar rot,Black shank,B,A
23,agrobench_png_975,Potato Late Blight,Powdery scab,B,D
26,agrobench_png_135,Leaf drop,Head rot,C,D
27,agrobench_png_1120,Bacterial blight,Bacterial leaf blight,C,A
30,agrobench_png_87,Leaf spot,Bois noir disease,B,A


In [21]:
import io
import torch
from PIL import Image

def predict_open_ended(image):
    prompt = (
        "Identify the plant disease visible in this image. "
        "Respond with only the most likely disease name. "
        "Do not provide any explanation."
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        if isinstance(v, torch.Tensor)
        else v
        for k, v in inputs.items()
    }

    model.eval()

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False
        )

    generated_ids_trimmed = generated_ids[
        :,
        inputs["input_ids"].shape[1]:
    ]

    prediction = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()

    return prediction

In [22]:
open_ended_results = []

for i in range(20):
    sample = test_dataset[i]

    ground_truth = sample["messages"][1]["content"][0]["text"]

    image = Image.open(
        io.BytesIO(sample["images"][0]["bytes"])
    ).convert("RGB")

    prediction = predict_open_ended(image)

    open_ended_results.append({
        "id": sample["id"],
        "ground_truth": ground_truth,
        "prediction": prediction
    })

    print(f"\nSample {i+1}")
    print("Ground Truth:", ground_truth)
    print("Prediction  :", prediction)
    print("-" * 50)


Sample 1
Ground Truth: Witches'-broom
Prediction  : blight
--------------------------------------------------

Sample 2
Ground Truth: Fire blight
Prediction  : blight
--------------------------------------------------

Sample 3
Ground Truth: Fire blight
Prediction  : Bacterial canker
--------------------------------------------------

Sample 4
Ground Truth: Mosaic
Prediction  : fig blight
--------------------------------------------------

Sample 5
Ground Truth: Phytophthora blight
Prediction  : Bacterial spot
--------------------------------------------------

Sample 6
Ground Truth: Anthracnose and charcoal spot
Prediction  : bacterial spot
--------------------------------------------------

Sample 7
Ground Truth: Crown gall
Prediction  : root rot
--------------------------------------------------

Sample 8
Ground Truth: Bacterial shoot blight
Prediction  : Bacterial spot
--------------------------------------------------

Sample 9
Ground Truth: Bacterial wilt
Prediction  : damping o

In [23]:
correct = 0

for result in open_ended_results:
    if (
        result["prediction"].strip().casefold()
        == result["ground_truth"].strip().casefold()
    ):
        correct += 1

accuracy = correct / len(open_ended_results) * 100

print("Correct:", correct)
print("Total:", len(open_ended_results))
print(f"Exact-Match Accuracy: {accuracy:.2f}%")

Correct: 0
Total: 20
Exact-Match Accuracy: 0.00%


In [24]:
csv_path = "/kaggle/working/qwen3vl_did_controlled_r16_2epoch_eval_301.csv"

df_controlled_r16.to_csv(
    csv_path,
    index=False
)

print("Saved successfully:")
print(csv_path)

Saved successfully:
/kaggle/working/qwen3vl_did_controlled_r16_2epoch_eval_301.csv


In [25]:
import os

print("Exists:", os.path.exists(csv_path))
print("Size:", os.path.getsize(csv_path), "bytes")

Exists: True
Size: 17820 bytes


In [1]:
import os

adapter_path = "/kaggle/input/datasets/iftekharackerman/qwen3vl-agrobench-did-lora-r16-2epochs"

print(os.listdir(adapter_path))

['adapter_model.safetensors', 'adapter_config.json', 'README.md', 'tokenizer.json', 'checkpoint-602', 'tokenizer_config.json', 'checkpoint-301', 'chat_template.jinja', 'processor_config.json']


In [2]:
!pip install -q --upgrade torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.1 MB/s eta 0:00:00a 0:00:01


In [3]:
import torch
import transformers
import peft
import torchao

print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT        :", peft.__version__)
print("TorchAO     :", torchao.__version__)

print(
    "GPU         :",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "No GPU"
)

PyTorch     : 2.10.0+cu128
Transformers: 5.0.0
PEFT        : 0.19.1
TorchAO     : 0.16.0
GPU         : Tesla T4


In [4]:
import torch

from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor
)

model_id = "Qwen/Qwen3-VL-2B-Instruct"

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

print("Base Qwen3-VL model loaded!")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Base Qwen3-VL model loaded!


In [5]:
adapter_path = "/kaggle/input/datasets/iftekharackerman/qwen3vl-agrobench-did-lora-r16-2epochs"

processor = AutoProcessor.from_pretrained(
    adapter_path
)

print("Processor loaded!")

Processor loaded!


In [6]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

model.eval()

print("Final LoRA adapter restored successfully!")

Final LoRA adapter restored successfully!


In [7]:
print(type(model))

print("\nActive adapter:")
print(model.active_adapter)

print("\nAdapter configuration:")
print(model.peft_config)

<class 'peft.peft_model.PeftModelForCausalLM'>

Active adapter:
default

Adapter configuration:
{'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path='Qwen/Qwen3-VL-2B-Instruct', revision=None, inference_mode=True, r=16, target_modules={'q_proj', 'o_proj', 'v_proj', 'k_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arro

In [9]:
from datasets import load_dataset
import json

dataset = load_dataset(
    "Project-AgML/AgroBench",
    split="train"
)

did_dataset = dataset.filter(
    lambda x:
    json.loads(x["raw_metadata"]).get("source") == "did"
)

did_split = did_dataset.train_test_split(
    test_size=0.20,
    seed=42
)

train_dataset = did_split["train"]
test_dataset = did_split["test"]

print("DID total:", len(did_dataset))
print("Train    :", len(train_dataset))
print("Test     :", len(test_dataset))

print("\nFirst train ID:", train_dataset[0]["id"])
print("First test ID :", test_dataset[0]["id"])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/650M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/840M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/629M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4342 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4342 [00:00<?, ? examples/s]

DID total: 1502
Train    : 1201
Test     : 301

First train ID: agrobench_png_169
First test ID : agrobench_png_1236


In [10]:
import re

def extract_candidate_diseases(question):
    matches = re.findall(
        r'[A-E]\.\s*(.+?)(?=\n[A-E]\.|$)',
        question,
        re.DOTALL
    )
    return [x.strip() for x in matches]

In [11]:
import torch

def predict_disease_from_candidates(
    image,
    crop_name,
    candidate_diseases
):
    option_letters = ["A", "B", "C", "D", "E"]

    options_text = "\n".join(
        f"{option_letters[i]}. {disease}"
        for i, disease in enumerate(candidate_diseases)
    )

    prompt = f"""
Crop: {crop_name}

Identify the disease visible in the image.

Choose the correct disease from the options below:

{options_text}

Respond with only the exact disease name from the selected option.
Do not provide any explanation.
""".strip()

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        if isinstance(v, torch.Tensor)
        else v
        for k, v in inputs.items()
    }

    model.eval()

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False
        )

    generated_ids_trimmed = generated_ids[
        :,
        inputs["input_ids"].shape[1]:
    ]

    prediction = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()

    return prediction

In [12]:
import io
from PIL import Image

for idx in range(10):
    sample = test_dataset[idx]

    question = sample["messages"][0]["content"][1]["text"]
    ground_truth = sample["messages"][1]["content"][0]["text"]

    candidate_diseases = extract_candidate_diseases(question)

    image = Image.open(
        io.BytesIO(sample["images"][0]["bytes"])
    ).convert("RGB")

    prediction = predict_disease_from_candidates(
        image=image,
        crop_name="Unknown",
        candidate_diseases=candidate_diseases
    )

    print(f"\nSample {idx + 1}")
    print("Candidates   :", candidate_diseases)
    print("Ground Truth :", ground_truth)
    print("Prediction   :", prediction)
    print("-" * 60)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Sample 1
Candidates   : ['Alfalfa dwarf', "Witches'-broom", 'Alfalfa mosaic', 'Phytophthora root and stem rot', 'Common leaf spot']
Ground Truth : Witches'-broom
Prediction   : Witches'-broom
------------------------------------------------------------

Sample 2
Candidates   : ['Phytophthora root rot', 'Spur blight', 'Raspberry ringspot', 'Yellow rust', 'Fire blight']
Ground Truth : Fire blight
Prediction   : Fire blight
------------------------------------------------------------

Sample 3
Candidates   : ['Alfalfa dwarf', 'Fire blight', 'Bacterial blast', 'Bacterial canker and blast', 'Pear decline']
Ground Truth : Fire blight
Prediction   : Fire blight
------------------------------------------------------------

Sample 4
Candidates   : ['Bacterial leaf blight and rot', 'Gummy stem blight', 'Lethal yellowing', 'Bacterial blight', 'Mosaic']
Ground Truth : Mosaic
Prediction   : Bacterial leaf blight and rot
------------------------------------------------------------

Sample 5
Candida

In [13]:
import json
from pprint import pprint

for idx in range(5):
    sample = did_dataset[idx]

    metadata = json.loads(sample["raw_metadata"])

    print(f"\n========== SAMPLE {idx + 1} ==========")
    print("ID:", sample["id"])
    print("Ground Truth:", sample["messages"][1]["content"][0]["text"])
    print("\nRaw metadata:")
    pprint(metadata)


========== SAMPLE 1 ==========
ID: agrobench_png_1
Ground Truth: Bacterial dieback of nectarine

Raw metadata:
{'category': 'Bacteria',
 'crop': 'Nectarine',
 'file_names': ['images/did_729.png'],
 'id': '729',
 'source': 'did'}

========== SAMPLE 2 ==========
ID: agrobench_png_2
Ground Truth: Phytophthora heart and root rot

Raw metadata:
{'category': 'Fungus',
 'crop': 'Pineapple',
 'file_names': ['images/did_683.png'],
 'id': '683',
 'source': 'did'}

========== SAMPLE 3 ==========
ID: agrobench_png_3
Ground Truth: Bacterial canker

Raw metadata:
{'category': 'Bacteria',
 'crop': 'Peach',
 'file_names': ['images/did_379.png'],
 'id': '379',
 'source': 'did'}

========== SAMPLE 4 ==========
ID: agrobench_png_4
Ground Truth: Bacterial leaf streak

Raw metadata:
{'category': 'Bacteria',
 'crop': 'Sorghum',
 'file_names': ['images/did_396.png'],
 'id': '396',
 'source': 'did'}

========== SAMPLE 5 ==========
ID: agrobench_png_5
Ground Truth: Bacterial fruit rot

Raw metadata:
{'categor

In [14]:
crop_values = []

for sample in did_dataset:
    metadata = json.loads(sample["raw_metadata"])
    crop_values.append(metadata.get("crop"))

print("Total DID samples :", len(crop_values))
print("Crop present      :", sum(c is not None for c in crop_values))
print("Crop missing      :", sum(c is None for c in crop_values))

unique_crops = sorted(
    set(c for c in crop_values if c is not None)
)

print("Unique crops      :", len(unique_crops))
print("\nFirst 30 crops:")
print(unique_crops[:30])

Total DID samples : 1502
Crop present      : 1502
Crop missing      : 0
Unique crops      : 157

First 30 crops:
['Alfalfa', 'Almond', 'Aloe', 'Amaranth', 'Apple', 'Apricot', 'Artichoke', 'Arugula', 'Asparagus', 'Avocado', 'Banana', 'Barley', 'Basil', 'Bean', 'Beet', 'Blackberry', 'Blueberry', 'Bougainvillea', 'Broccoli', 'Butternut', 'Cabbage', 'Cactus', 'Campanula', 'Cantaloupe', 'Carrot', 'Cassava', 'Casuarina', 'Cauliflower', 'Celery', 'Chard']


In [15]:
from collections import defaultdict
import json

crop_to_diseases = defaultdict(set)

for sample in did_dataset:
    metadata = json.loads(sample["raw_metadata"])
    crop = metadata["crop"]

    disease = sample["messages"][1]["content"][0]["text"].strip()

    crop_to_diseases[crop].add(disease)

# Convert sets to sorted lists
crop_to_diseases = {
    crop: sorted(list(diseases))
    for crop, diseases in crop_to_diseases.items()
}

print("Total crops:", len(crop_to_diseases))

Total crops: 157


In [16]:
for crop in ["Apple", "Tomato", "Potato", "Rice", "Banana"]:
    print(f"\n{crop}")
    print("-" * 40)

    diseases = crop_to_diseases.get(crop, [])

    for disease in diseases:
        print(disease)

    print("Total diseases:", len(diseases))


Apple
----------------------------------------
Apple proliferation
Apple scab
Black rot
Blister spot
Blossom blast
Cedar apple rust
Crown gall
Fire blight
Flyspeck
Hairy root
Phytophthora crown and root rot
Powdery mildew
Total diseases: 12

Tomato
----------------------------------------
Anthracnose
Bacterial canker
Bacterial fruit rot
Bacterial speck
Bacterial spot
Bacterial stem rot
Bacterial wilt
Black mold
Blossom-end rot
Buckeye rot
Catface
Early blight
Edema
Fusarium wilt
Gray mold
Late blight
Leaf Mold
Little leaf
Magnesium deficiency
Pith necrosis
Septoria leaf spot
Sunscald
Syringae leaf spot
Target Spot
Tomato Yellow Leaf Curl disease
Tomato big bud
Tomato mosaic virus
Tomato spotted wilt
Verticillium wilt
Water stress
Zippering
Total diseases: 31

Potato
----------------------------------------
Aster yellows
Bacterial ring rot
Bacterial soft rot
Black scurf & Rhizoctonia canker
Blackleg
Brown rot
Common scab
Potato Early Blight
Potato Late Blight
Potato leaf roll
Potato vi

In [17]:
crop_disease_counts = {
    crop: len(diseases)
    for crop, diseases in crop_to_diseases.items()
}

sorted_counts = sorted(
    crop_disease_counts.items(),
    key=lambda x: x[1],
    reverse=True
)

print("Top 30 crops with most diseases:\n")

for crop, count in sorted_counts[:30]:
    print(f"{crop:20s} -> {count}")

Top 30 crops with most diseases:

Tomato               -> 31
Maize                -> 25
Onion                -> 19
Rice                 -> 17
Cantaloupe           -> 15
Strawberry           -> 14
Peach                -> 13
Papaya               -> 13
Bean                 -> 13
Barley               -> 13
Beet                 -> 13
Cucumber             -> 13
Potato               -> 13
Apple                -> 12
Tobacco              -> 12
Banana               -> 12
Pepper               -> 11
Wheat                -> 11
Raspberry            -> 10
Peanut               -> 10
Alfalfa              -> 10
Grape                -> 9
Almond               -> 9
Apricot              -> 9
Lime                 -> 9
Cherry               -> 9
Lettuce              -> 8
Watermelon           -> 8
Turnip               -> 8
Cabbage              -> 8


In [18]:
counts = list(crop_disease_counts.values())

print("Minimum diseases per crop :", min(counts))
print("Maximum diseases per crop :", max(counts))

print("Crops with <= 5 diseases :", sum(x <= 5 for x in counts))
print("Crops with > 5 diseases  :", sum(x > 5 for x in counts))
print("Crops with > 10 diseases :", sum(x > 10 for x in counts))

Minimum diseases per crop : 1
Maximum diseases per crop : 31
Crops with <= 5 diseases : 116
Crops with > 5 diseases  : 41
Crops with > 10 diseases : 18


In [19]:
from collections import defaultdict
import json

crop_category_diseases = defaultdict(lambda: defaultdict(set))

for sample in did_dataset:
    metadata = json.loads(sample["raw_metadata"])

    crop = metadata["crop"]
    category = metadata["category"]

    disease = sample["messages"][1]["content"][0]["text"].strip()

    crop_category_diseases[crop][category].add(disease)

# Convert sets to sorted lists
crop_category_diseases = {
    crop: {
        category: sorted(list(diseases))
        for category, diseases in categories.items()
    }
    for crop, categories in crop_category_diseases.items()
}

In [20]:
all_categories = sorted({
    category
    for categories in crop_category_diseases.values()
    for category in categories
})

print("Categories:")
for category in all_categories:
    print("-", category)

Categories:
- Algae
- Bacteria
- Environmental
- Fungus
- Nutritional
- Parasitic
- Phytoplasma
- Virus


In [21]:
for crop in ["Tomato", "Maize", "Rice", "Potato", "Apple", "Banana"]:
    print(f"\n{'='*60}")
    print(crop)
    print("="*60)

    for category, diseases in crop_category_diseases[crop].items():
        print(f"\n{category} ({len(diseases)} diseases)")
        for disease in diseases:
            print("  -", disease)


Tomato

Bacteria (8 diseases)
  - Bacterial canker
  - Bacterial fruit rot
  - Bacterial speck
  - Bacterial spot
  - Bacterial stem rot
  - Bacterial wilt
  - Pith necrosis
  - Syringae leaf spot

Fungus (11 diseases)
  - Anthracnose
  - Black mold
  - Buckeye rot
  - Early blight
  - Fusarium wilt
  - Gray mold
  - Late blight
  - Leaf Mold
  - Septoria leaf spot
  - Target Spot
  - Verticillium wilt

Virus (3 diseases)
  - Tomato Yellow Leaf Curl disease
  - Tomato mosaic virus
  - Tomato spotted wilt

Environmental (5 diseases)
  - Catface
  - Edema
  - Sunscald
  - Water stress
  - Zippering

Phytoplasma (2 diseases)
  - Little leaf
  - Tomato big bud

Nutritional (2 diseases)
  - Blossom-end rot
  - Magnesium deficiency

Maize

Bacteria (9 diseases)
  - Bacterial Leaf Streak disease
  - Bacterial leaf blight and stalk rot
  - Bacterial stalk and top rot
  - Bacterial stripe
  - Goss's bacterial blight
  - Goss's bacterial wilt
  - Holcus spot
  - Stewart's wilt
  - soft rot

Fun

In [22]:
import torch

def predict_from_crop_candidates(image, crop_name, candidate_diseases):

    options_text = "\n".join(
        f"{i+1}. {disease}"
        for i, disease in enumerate(candidate_diseases)
    )

    prompt = f"""
Crop: {crop_name}

Identify the disease visible in the plant image.

Choose the most likely disease from the candidate diseases below:

{options_text}

Respond with only the exact disease name from the list.
Do not provide any explanation.
""".strip()

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        if isinstance(v, torch.Tensor)
        else v
        for k, v in inputs.items()
    }

    model.eval()

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False
        )

    generated_ids_trimmed = generated_ids[
        :, inputs["input_ids"].shape[1]:
    ]

    prediction = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()

    return prediction

In [23]:
import io
import json
from PIL import Image

correct = 0

for idx in range(10):

    sample = test_dataset[idx]

    metadata = json.loads(sample["raw_metadata"])
    crop = metadata["crop"]

    ground_truth = (
        sample["messages"][1]["content"][0]["text"].strip()
    )

    candidate_diseases = crop_to_diseases[crop]

    image = Image.open(
        io.BytesIO(sample["images"][0]["bytes"])
    ).convert("RGB")

    prediction = predict_from_crop_candidates(
        image=image,
        crop_name=crop,
        candidate_diseases=candidate_diseases
    )

    is_correct = (
        prediction.casefold() ==
        ground_truth.casefold()
    )

    if is_correct:
        correct += 1

    print(f"\nSample {idx + 1}")
    print("Crop          :", crop)
    print("Candidates    :", len(candidate_diseases))
    print("Ground Truth  :", ground_truth)
    print("Prediction    :", prediction)
    print("Correct       :", "✓" if is_correct else "✗")
    print("-" * 60)

print(f"\nAccuracy: {correct}/10 = {correct/10:.2%}")


Sample 1
Crop          : Alfalfa
Candidates    : 10
Ground Truth  : Witches'-broom
Prediction    : Witches'-broom
Correct       : ✓
------------------------------------------------------------

Sample 2
Crop          : Raspberry
Candidates    : 10
Ground Truth  : Fire blight
Prediction    : Fire blight
Correct       : ✓
------------------------------------------------------------

Sample 3
Crop          : Pear
Candidates    : 4
Ground Truth  : Fire blight
Prediction    : Fire blight
Correct       : ✓
------------------------------------------------------------

Sample 4
Crop          : Fig
Candidates    : 1
Ground Truth  : Mosaic
Prediction    : Mosaic
Correct       : ✓
------------------------------------------------------------

Sample 5
Crop          : Pepper
Candidates    : 11
Ground Truth  : Phytophthora blight
Prediction    : Bacterial spot
Correct       : ✗
------------------------------------------------------------

Sample 6
Crop          : Papaya
Candidates    : 13
Ground Tr

In [24]:
def normalize_text(text):
    return " ".join(text.strip().casefold().split())


def match_to_candidate(prediction, candidates):
    pred_norm = normalize_text(prediction)

    # Exact match first
    for disease in candidates:
        if pred_norm == normalize_text(disease):
            return disease

    # Candidate contained in generated response
    for disease in candidates:
        disease_norm = normalize_text(disease)

        if disease_norm in pred_norm:
            return disease

    return None

In [25]:
import io
import json
import pandas as pd
from PIL import Image

results = []

for idx, sample in enumerate(test_dataset):

    metadata = json.loads(sample["raw_metadata"])
    crop = metadata["crop"]

    ground_truth = (
        sample["messages"][1]["content"][0]["text"].strip()
    )

    candidate_diseases = crop_to_diseases[crop]

    image = Image.open(
        io.BytesIO(sample["images"][0]["bytes"])
    ).convert("RGB")

    raw_prediction = predict_from_crop_candidates(
        image=image,
        crop_name=crop,
        candidate_diseases=candidate_diseases
    )

    matched_prediction = match_to_candidate(
        raw_prediction,
        candidate_diseases
    )

    is_correct = (
        matched_prediction is not None
        and normalize_text(matched_prediction)
        == normalize_text(ground_truth)
    )

    results.append({
        "id": sample["id"],
        "crop": crop,
        "num_candidates": len(candidate_diseases),
        "ground_truth": ground_truth,
        "raw_prediction": raw_prediction,
        "matched_prediction": matched_prediction,
        "correct": is_correct
    })

    if (idx + 1) % 25 == 0:
        print(f"Processed {idx + 1}/{len(test_dataset)}")

Processed 25/301
Processed 50/301
Processed 75/301
Processed 100/301
Processed 125/301
Processed 150/301
Processed 175/301
Processed 200/301
Processed 225/301
Processed 250/301
Processed 275/301
Processed 300/301


In [26]:
df_crop_eval = pd.DataFrame(results)

total = len(df_crop_eval)
correct = df_crop_eval["correct"].sum()
unmatched = df_crop_eval["matched_prediction"].isna().sum()

accuracy = correct / total

print("===== CROP-BASED DEPLOYMENT EVALUATION =====")
print(f"Correct   : {correct}/{total}")
print(f"Accuracy  : {accuracy:.2%}")
print(f"Unmatched : {unmatched}")

===== CROP-BASED DEPLOYMENT EVALUATION =====
Correct   : 108/301
Accuracy  : 35.88%
Unmatched : 0


In [27]:
df_crop_eval["candidate_group"] = pd.cut(
    df_crop_eval["num_candidates"],
    bins=[0, 5, 10, 15, 100],
    labels=["1-5", "6-10", "11-15", "16+"]
)

group_results = (
    df_crop_eval
    .groupby("candidate_group", observed=True)
    .agg(
        samples=("correct", "size"),
        correct=("correct", "sum"),
        avg_candidates=("num_candidates", "mean")
    )
)

group_results["accuracy"] = (
    group_results["correct"] /
    group_results["samples"]
)

print(group_results)

                 samples  correct  avg_candidates  accuracy
candidate_group                                            
1-5                   89       57        3.269663  0.640449
6-10                  86       29        8.058140  0.337209
11-15                 90       18       12.711111  0.200000
16+                   36        4       24.888889  0.111111


In [28]:
import torch

DISEASE_CATEGORIES = [
    "Algae",
    "Bacteria",
    "Environmental",
    "Fungus",
    "Nutritional",
    "Parasitic",
    "Phytoplasma",
    "Virus"
]


def predict_disease_category(image, crop_name):

    categories_text = "\n".join(
        f"{i+1}. {category}"
        for i, category in enumerate(DISEASE_CATEGORIES)
    )

    prompt = f"""
Crop: {crop_name}

Examine the plant image and determine the most likely category
of the plant problem.

Choose exactly one category from the following list:

{categories_text}

Respond with only the exact category name from the list.
Do not provide any explanation.
""".strip()

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        if isinstance(v, torch.Tensor)
        else v
        for k, v in inputs.items()
    }

    model.eval()

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    generated_ids_trimmed = generated_ids[
        :, inputs["input_ids"].shape[1]:
    ]

    prediction = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()

    return prediction

In [29]:
import io
import json
from PIL import Image

category_correct = 0

for idx in range(20):

    sample = test_dataset[idx]

    metadata = json.loads(sample["raw_metadata"])

    crop = metadata["crop"]
    true_category = metadata["category"]

    image = Image.open(
        io.BytesIO(sample["images"][0]["bytes"])
    ).convert("RGB")

    prediction = predict_disease_category(
        image=image,
        crop_name=crop
    )

    is_correct = (
        prediction.strip().casefold()
        == true_category.strip().casefold()
    )

    if is_correct:
        category_correct += 1

    print(f"\nSample {idx + 1}")
    print("Crop          :", crop)
    print("True Category :", true_category)
    print("Prediction    :", prediction)
    print("Correct       :", "✓" if is_correct else "✗")
    print("-" * 50)

print(
    f"\nCategory Accuracy: "
    f"{category_correct}/20 = {category_correct/20:.2%}"
)


Sample 1
Crop          : Alfalfa
True Category : Phytoplasma
Prediction    : Parasitic
Correct       : ✗
--------------------------------------------------

Sample 2
Crop          : Raspberry
True Category : Bacteria
Prediction    : Fungus
Correct       : ✗
--------------------------------------------------

Sample 3
Crop          : Pear
True Category : Bacteria
Prediction    : Environmental
Correct       : ✗
--------------------------------------------------

Sample 4
Crop          : Fig
True Category : Virus
Prediction    : Virus
Correct       : ✓
--------------------------------------------------

Sample 5
Crop          : Pepper
True Category : Fungus
Prediction    : Fungus
Correct       : ✓
--------------------------------------------------

Sample 6
Crop          : Papaya
True Category : Fungus
Prediction    : Fungus
Correct       : ✓
--------------------------------------------------

Sample 7
Crop          : Kiwi
True Category : Bacteria
Prediction    : Parasitic
Correct       

In [30]:
import json

for idx in range(20):

    sample = test_dataset[idx]

    metadata = json.loads(sample["raw_metadata"])

    crop = metadata["crop"]
    category = metadata["category"]
    ground_truth = sample["messages"][1]["content"][0]["text"].strip()

    question = sample["messages"][0]["content"][1]["text"]
    candidates = extract_candidate_diseases(question)

    print(f"\n{'='*70}")
    print(f"Sample {idx + 1}")
    print(f"Crop        : {crop}")
    print(f"Category    : {category}")
    print(f"Ground Truth: {ground_truth}")
    print("Candidates:")

    for candidate in candidates:
        marker = " <-- CORRECT" if candidate == ground_truth else ""
        print(f"  - {candidate}{marker}")


Sample 1
Crop        : Alfalfa
Category    : Phytoplasma
Ground Truth: Witches'-broom
Candidates:
  - Alfalfa dwarf
  - Witches'-broom <-- CORRECT
  - Alfalfa mosaic
  - Phytophthora root and stem rot
  - Common leaf spot

Sample 2
Crop        : Raspberry
Category    : Bacteria
Ground Truth: Fire blight
Candidates:
  - Phytophthora root rot
  - Spur blight
  - Raspberry ringspot
  - Yellow rust
  - Fire blight <-- CORRECT

Sample 3
Crop        : Pear
Category    : Bacteria
Ground Truth: Fire blight
Candidates:
  - Alfalfa dwarf
  - Fire blight <-- CORRECT
  - Bacterial blast
  - Bacterial canker and blast
  - Pear decline

Sample 4
Crop        : Fig
Category    : Virus
Ground Truth: Mosaic
Candidates:
  - Bacterial leaf blight and rot
  - Gummy stem blight
  - Lethal yellowing
  - Bacterial blight
  - Mosaic <-- CORRECT

Sample 5
Crop        : Pepper
Category    : Fungus
Ground Truth: Phytophthora blight
Candidates:
  - Southern blight
  - Damping-off
  - Phytophthora blight <-- CORRE

In [31]:
import json

mapping_path = "/kaggle/working/crop_to_diseases.json"

with open(mapping_path, "w", encoding="utf-8") as f:
    json.dump(
        crop_to_diseases,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", mapping_path)
print("Total crops:", len(crop_to_diseases))

Saved: /kaggle/working/crop_to_diseases.json
Total crops: 157


In [32]:
with open(mapping_path, "r", encoding="utf-8") as f:
    loaded_mapping = json.load(f)

print("Crops:", len(loaded_mapping))

print("\nPotato:")
print(loaded_mapping["Potato"])

print("\nTomato diseases:", len(loaded_mapping["Tomato"]))

Crops: 157

Potato:
['Aster yellows', 'Bacterial ring rot', 'Bacterial soft rot', 'Black scurf & Rhizoctonia canker', 'Blackleg', 'Brown rot', 'Common scab', 'Potato Early Blight', 'Potato Late Blight', 'Potato leaf roll', 'Potato virus Y', 'Powdery scab', 'Verticillium wilt']

Tomato diseases: 31


In [34]:
csv_path = "/kaggle/working/qwen3vl_crop_based_deployment_eval_301.csv"

df_crop_eval.to_csv(csv_path, index=False)

print("Saved:", csv_path)
print("Rows:", len(df_crop_eval))
print("Columns:", df_crop_eval.columns.tolist())

Saved: /kaggle/working/qwen3vl_crop_based_deployment_eval_301.csv
Rows: 301
Columns: ['id', 'crop', 'num_candidates', 'ground_truth', 'raw_prediction', 'matched_prediction', 'correct', 'candidate_group']


In [35]:
import os

print("Exists:", os.path.exists(csv_path))
print("Size:", os.path.getsize(csv_path), "bytes")

Exists: True
Size: 25413 bytes
